# 09 — SAM 3 en Video

## ¿Qué vamos a construir hoy?

Procesarás un video con segmentación precisa frame por frame usando dos pipelines:

- **Opción A — YOLO + SAM 3:** detección con YOLO, segmentación con SAM, tracking con ByteTrack.
- **Opción B — Solo SAM 3:** texto directo como prompt, sin detector previo.

**Aprenderás a:**
- Mantener el pipeline YOLO → ByteTrack → SAM 3 (Opción A)
- Usar `SAM3VideoSemanticPredictor` con texto para video (Opción B)
- Entender las ventajas y limitaciones de cada enfoque en video

**Tiempo estimado:** 45 minutos

## ⏱️ Estructura de la Clase (Duración estimada: 1 hora)
- **Introducción y Conceptos Base**: 15 min
- **Desarrollo y Demostración Práctica**: 25 min
- **Análisis y Casos Extremos (Pausa y Observa)**: 20 min

## Dos pipelines para video

**Opción A — YOLO + ByteTrack + SAM 3** (NB04 + NB06 + nuevo):

```
Frame → YOLO → ByteTrack → SAM 3 → MaskAnnotator
          ↑        ↑          ↑
       detecta  asigna ID  genera silueta exacta
```

**Opción B — SAM 3 directo con texto** (SAM 3 nuevo):

```
Frame → SAM3VideoSemanticPredictor(text=["car","bus"]) → MaskAnnotator
                    ↑
          detecta, segmenta y rastrea en un solo paso
```

La Opción A da más control (puedes filtrar por clase, usar tu propio detector).
La Opción B es más simple: un modelo, un prompt de texto, sin pasos intermedios.

**Nota de rendimiento:** SAM 3 es más lento que YOLO por frame.
En GPU es viable para tiempo real; en CPU, el procesado offline funciona perfectamente.


In [ ]:
%pip install supervision ultralytics trackers
%pip install -q rfdetr "trackers==2.4.0"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
sam_path = "/content/drive/MyDrive/RandD/Archive_Zero_Resolved/sam3.pt"

In [ ]:
import supervision as sv
from ultralytics import YOLO, SAM
from trackers import ByteTrackTracker
import cv2
import numpy as np

# =================================================================
# PARCHE GLOBAL NUMPY 2.0 PARA SUPERVISION (Geometría)
_original_cross = np.cross

def cross_patched(a, b, axisa=-1, axisb=-1, axisc=-1, axis=None):
    a = np.asarray(a)
    b = np.asarray(b)
    if a.shape[-1] == 2 and b.shape[-1] == 2:
        return a[..., 0] * b[..., 1] - a[..., 1] * b[..., 0]
    return _original_cross(a, b, axisa, axisb, axisc, axis)

np.cross = cross_patched
# =================================================================

import urllib.request
from pathlib import Path

Path("assets").mkdir(exist_ok=True)
urllib.request.urlretrieve(
    "https://media.roboflow.com/supervision/video-examples/vehicles.mp4",
    "assets/vehicles.mp4",
)

yolo_model = YOLO("yolov8n.pt")
sam_model  = SAM(sam_path)
# sam3.pt ≈ 3.4 GB — si no se descarga automáticamente:
# https://huggingface.co/facebook/sam3
tracker    = ByteTrackTracker()

video_info = sv.VideoInfo.from_video_path("assets/vehicles.mp4")
print(f"Resolución: {video_info.width} × {video_info.height}")
print(f"FPS: {video_info.fps} | Total frames: {video_info.total_frames}")

## El pipeline completo: YOLO + ByteTrack + SAM


In [ ]:
mask_annotator  = sv.MaskAnnotator(opacity=0.6)
label_annotator = sv.LabelAnnotator()
trace_annotator = sv.TraceAnnotator()

tracker = ByteTrackTracker()

def procesar_frame_sam(frame: np.ndarray, _: int) -> np.ndarray:
    # Paso 1: Detectar con YOLO y asignar IDs persistentes
    yolo_results = yolo_model(frame, verbose=False)[0]
    yolo_det     = sv.Detections.from_ultralytics(yolo_results)
    yolo_det     = tracker.update(yolo_det)
    
    if len(yolo_det) == 0:
        return frame
    
    # Paso 2: SAM genera máscaras usando las cajas de YOLO como guía
    # .tolist() es necesario porque SAM espera lista de Python, no array NumPy
    bboxes      = yolo_det.xyxy.tolist()
    sam_results = sam_model(frame, bboxes=bboxes, verbose=False)[0]
    sam_det     = sv.Detections.from_ultralytics(sam_results)
    
    # Paso 3: Transferir atributos de YOLO a SAM
    # SAM preserva el orden de los bboxes de entrada — la alineación posicional es segura
    if len(sam_det) == len(yolo_det):
        sam_det.tracker_id = yolo_det.tracker_id
        sam_det.class_id   = yolo_det.class_id
        sam_det.confidence = yolo_det.confidence
    
    labels = [
        f"ID:{tid} {yolo_results.names[c]}"
        for tid, c in zip(sam_det.tracker_id, sam_det.class_id)
    ]
    
    annotated = mask_annotator.annotate(scene=frame.copy(), detections=sam_det)
    annotated = label_annotator.annotate(scene=annotated, detections=sam_det, labels=labels)
    annotated = trace_annotator.annotate(scene=annotated, detections=sam_det)
    return annotated

sv.process_video(
    source_path="assets/vehicles.mp4",
    target_path="assets/vehicles_sam.mp4",
    callback=procesar_frame_sam,
    show_progress = True
)
print("Guardado: assets/vehicles_sam.mp4")

In [ ]:
from google.colab import files
files.download("assets/vehicles_sam.mp4")

## Pausa y observa: atributos antes y después de la transferencia


In [ ]:
# Inspeccionamos un frame para ver qué contiene sam_det ANTES y DESPUÉS de la transferencia
tracker = ByteTrackTracker()
cap = cv2.VideoCapture("assets/vehicles.mp4")
ret, frame_test = cap.read()
cap.release()

yolo_r   = yolo_model(frame_test, verbose=False)[0]
yolo_det = sv.Detections.from_ultralytics(yolo_r)
yolo_det = tracker.update(yolo_det)

bboxes   = yolo_det.xyxy.tolist()
sam_r    = sam_model(frame_test, bboxes=bboxes, verbose=False)[0]
sam_det  = sv.Detections.from_ultralytics(sam_r)

print("SAM ANTES de transferencia:")
print(f"  tracker_id: {sam_det.tracker_id}")   # None — SAM no sabe de tracking
print(f"  class_id:   {sam_det.class_id}")      # None o índice SAM interno
print(f"  mask shape: {sam_det.mask.shape if sam_det.mask is not None else None}")

if len(sam_det) == len(yolo_det):
    sam_det.tracker_id = yolo_det.tracker_id
    sam_det.class_id   = yolo_det.class_id
    sam_det.confidence = yolo_det.confidence

print("\nSAM DESPUÉS de transferencia:")
print(f"  tracker_id: {sam_det.tracker_id}")   # ahora tiene IDs de ByteTrack
print(f"  class_id:   {sam_det.class_id}")      # ahora tiene clases de YOLO

## 🔧 Exploración interactiva

### Experimento 1: Segmentar solo objetos en una zona


In [ ]:
# Combinamos PolygonZone (NB05) con SAM (NB06/NB07)
# Solo segmentamos los objetos que están dentro de la zona
POLYGON = np.array([
    [0,                         video_info.height // 2],
    [video_info.width // 2,     video_info.height // 2],
    [video_info.width // 2,     video_info.height],
    [0,                         video_info.height],
])
zone     = sv.PolygonZone(polygon=POLYGON)
zone_ann = sv.PolygonZoneAnnotator(zone=zone, color=sv.Color.RED, thickness=3)

tracker = ByteTrackTracker()

def callback_zona_sam(frame: np.ndarray, _: int) -> np.ndarray:
    yolo_r   = yolo_model(frame, verbose=False)[0]
    yolo_det = sv.Detections.from_ultralytics(yolo_r)
    yolo_det = tracker.update(yolo_det)
    
    # Filtrar solo objetos en la zona ANTES de llamar a SAM
    # — SAM es lento: no gastar tiempo en objetos fuera de interés
    en_zona  = zone.trigger(detections=yolo_det)
    yolo_det = yolo_det[en_zona]
    
    annotated = frame.copy()
    if len(yolo_det) > 0:
        bboxes   = yolo_det.xyxy.tolist()
        sam_r    = sam_model(frame, bboxes=bboxes, verbose=False)[0]
        sam_det  = sv.Detections.from_ultralytics(sam_r)
        if len(sam_det) == len(yolo_det):
            sam_det.tracker_id = yolo_det.tracker_id
            sam_det.class_id   = yolo_det.class_id
        annotated = mask_annotator.annotate(scene=annotated, detections=sam_det)
    
    annotated = zone_ann.annotate(scene=annotated)
    return annotated

sv.process_video(
    source_path="assets/vehicles.mp4",
    target_path="assets/vehicles_sam_zona.mp4",
    callback=callback_zona_sam,
    show_progress=True
)
print("Guardado: assets/vehicles_sam_zona.mp4")
# 💭 Reflexión: ¿Cuánto más rápido es este callback comparado con el que segmenta todo?
# Al filtrar antes de SAM, reducimos el número de objetos que SAM procesa por frame.

### Experimento 2: Opacidad dinámica según la confianza de la detección


In [ ]:
# MaskAnnotator tiene una opacidad fija para todos los objetos.
# Para opacidad por objeto, podemos anotar objeto por objeto manualmente.
tracker = ByteTrackTracker()

def callback_opacidad(frame: np.ndarray, _: int) -> np.ndarray:
    yolo_r   = yolo_model(frame, verbose=False)[0]
    yolo_det = sv.Detections.from_ultralytics(yolo_r)
    yolo_det = tracker.update(yolo_det)
    
    if len(yolo_det) == 0:
        return frame
    
    bboxes   = yolo_det.xyxy.tolist()
    sam_r    = sam_model(frame, bboxes=bboxes, verbose=False)[0]
    sam_det  = sv.Detections.from_ultralytics(sam_r)
    if len(sam_det) == len(yolo_det):
        sam_det.tracker_id = yolo_det.tracker_id
        sam_det.confidence = yolo_det.confidence
    
    annotated = frame.copy()
    # Anotar objeto por objeto usando su confianza como opacidad
    for i in range(len(sam_det)):
        det_i    = sam_det[i]
        # Alta confianza → máscara más opaca (más "seguro" el objeto)
        opacidad = float(det_i.confidence[0]) if det_i.confidence is not None else 0.5
        annotated = sv.MaskAnnotator(opacity=opacidad).annotate(scene=annotated, detections=det_i)
    return annotated

sv.process_video(
    source_path="assets/vehicles.mp4",
    target_path="assets/vehicles_sam_opacidad.mp4",
    callback=callback_opacidad,
    show_progress=True
)
print("Guardado: assets/vehicles_sam_opacidad.mp4")
# 💭 Reflexión: ¿Qué objetos se ven con máscara más opaca?
# Los que el modelo detecta con más certeza — generalmente los más grandes y visibles.

### Experimento 3: Área de máscara por objeto a lo largo del tiempo


In [ ]:
# Rastreamos cómo cambia el área de la máscara de cada objeto frame a frame
# — útil para detectar cuándo un objeto se acerca o aleja de la cámara
areas_por_id = {}   # {tracker_id: [área_frame_0, área_frame_1, ...]}

tracker = ByteTrackTracker()

def callback_areas(frame: np.ndarray, frame_idx: int) -> np.ndarray:
    yolo_r   = yolo_model(frame, verbose=False)[0]
    yolo_det = sv.Detections.from_ultralytics(yolo_r)
    yolo_det = tracker.update(yolo_det)
    
    if len(yolo_det) == 0:
        return frame
    
    bboxes   = yolo_det.xyxy.tolist()
    sam_r    = sam_model(frame, bboxes=bboxes, verbose=False)[0]
    sam_det  = sv.Detections.from_ultralytics(sam_r)
    if len(sam_det) == len(yolo_det):
        sam_det.tracker_id = yolo_det.tracker_id
    
    # Registrar el área de cada objeto en este frame
    if sam_det.mask is not None:
        for i in range(len(sam_det)):
            tid  = sam_det.tracker_id[i]
            area = int(sam_det.mask[i].sum())  # número de píxeles True
            if tid not in areas_por_id:
                areas_por_id[tid] = []
            areas_por_id[tid].append(area)
    
    return mask_annotator.annotate(scene=frame.copy(), detections=sam_det)

sv.process_video(
    source_path="assets/vehicles.mp4",
    target_path="assets/vehicles_sam_areas.mp4",
    callback=callback_areas,
    show_progress=True
)

# Mostrar evolución del área de los 3 objetos con más frames registrados
import matplotlib.pyplot as plt
ids_con_datos = sorted(areas_por_id, key=lambda k: len(areas_por_id[k]), reverse=True)[:3]
plt.figure(figsize=(12, 4))
for tid in ids_con_datos:
    plt.plot(areas_por_id[tid], label=f"ID {tid}")
plt.xlabel("Frame")
plt.ylabel("Área de máscara (px²)")
plt.title("Evolución del área de máscara por objeto")
plt.legend()
plt.tight_layout()
plt.show()
# 💭 Reflexión: ¿Qué evento físico causa que el área aumente o disminuya?
# Un aumento sostenido → el objeto se acerca a la cámara.
# Una disminución → se aleja. Un cambio brusco → oclusión parcial o nueva detección.

---

## Opción B: SAM 3 directo con texto — sin YOLO

`SAM3VideoSemanticPredictor` procesa el video frame a frame usando un prompt
de texto. SAM 3 detecta, segmenta y rastrea las instancias del concepto
indicado en un solo paso — sin necesidad de YOLO ni ByteTrack.


In [ ]:
from ultralytics.models.sam import SAM3VideoSemanticPredictor
import torch

overrides = dict(conf=0.25, task="segment", mode="predict", model="sam3.pt")
if torch.cuda.is_available():
    overrides["half"] = True

video_predictor = SAM3VideoSemanticPredictor(overrides=overrides)

# stream=True procesa frame a frame sin cargar todo el video en memoria
resultados_video = video_predictor(
    source="assets/vehicles.mp4",
    text=["car", "bus", "truck"],
    stream=True
)

# Convertir y guardar frame a frame
import cv2
from pathlib import Path

cap    = cv2.VideoCapture("assets/vehicles.mp4")
fps    = cap.get(cv2.CAP_PROP_FPS)
w      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

writer = cv2.VideoWriter(
    "assets/vehicles_texto.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h)
)

mask_ann = sv.MaskAnnotator(opacity=0.6)

for res in resultados_video:
    det      = sv.Detections.from_ultralytics(res)
    frame    = res.orig_img
    annotado = mask_ann.annotate(scene=frame.copy(), detections=det)
    writer.write(annotado)

writer.release()
print("Guardado: assets/vehicles_texto.mp4")


### Comparación: Opción A vs. Opción B

| | Opción A (YOLO + SAM 3) | Opción B (solo SAM 3) |
|---|---|---|
| **Detector previo** | Necesitas YOLO | No necesitas |
| **Prompt** | Bounding boxes | Texto libre |
| **Tracking** | ByteTrack (IDs estables) | SAM3 interno |
| **Velocidad** | Más lento (2 modelos) | Más rápido (1 modelo) |
| **Control** | Alto (filtrar por clase, umbral YOLO) | Bajo (depende del texto) |
| **Mejor para** | Dominios con detector entrenado | Exploración rápida |


## 🚀 Reto de extensión

**Tarea:** Procesa el video segmentando solo los vehículos cuyo `tracker_id` sea
menor o igual a 5 (los 5 primeros en aparecer). Ignora los demás.

**Pista:**
```python
# Después de tracker.update(yolo_det):
primeros = yolo_det[yolo_det.tracker_id <= 5]
# Continúa el pipeline con primeros en lugar de yolo_det
```

¿Qué pasa cuando los primeros 5 objetos salen del frame?
¿El video se queda sin máscaras?

In [ ]:
# Escribe tu solución aquí
tracker = ByteTrackTracker()

def mi_callback(frame: np.ndarray, _: int) -> np.ndarray:
    yolo_r   = yolo_model(frame, verbose=False)[0]
    yolo_det = sv.Detections.from_ultralytics(yolo_r)
    yolo_det = tracker.update(yolo_det)
    mask_valida = yolo_det.tracker_id <= 5        # máscara booleana: primeros 5 IDs
    primeros     = yolo_det[mask_valida]           # sv.Detections filtrado
    if len(primeros) == 0:
        return frame.copy()
    # sam_r = predictor(bboxes=primeros.xyxy)[0]
    # sam_det = sv.Detections.from_ultralytics(sam_r)
    # annotated = mask_annotator.annotate(scene=frame.copy(), detections=sam_det)
    # return annotated
    return frame.copy()